# Ders 12 — VLANeXt'i ücretsiz T4'te LIBERO üzerinde çalıştır

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Kaggle'daysan Settings → Accelerator → **GPU T4 x2** (RAM 29 GB; Colab'ın 12.7 GB'ına göre daha güvenli).

Akış: (1) donanımı doğrula → (2) sistem kütüphaneleri → (3) VLANeXt + LIBERO kur → (4) T4 yamaları → (5) checkpoint indir (16.4 GB) → (6) 2–3 episode eval → (7) sonucu oku, videoyu izle, `runs.csv` satırı üret.

Toplam süre (hipotez, T4): kurulum ~10 dk + indirme ~5 dk + 3 episode ~5 dk. Hücreleri sırayla çalıştır; hata alırsan hücre çıktısının **tamamını** chat'e yapıştır.

In [ ]:
# 1) Donanım kontrolü — compute capability 7.5 = T4 (FlashAttention-2 YOK, bf16 donanım desteği YOK → yamalar gerekli)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk boş %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))
print("bf16 (emülasyon dahil):", torch.cuda.is_bf16_supported())

In [ ]:
# 2) MuJoCo'nun ekransız (EGL) render için istediği sistem kütüphaneleri — VLANeXt COMMON_ISSUES #1
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 > /dev/null
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"
print("ok")

In [ ]:
# 3) VLANeXt + LIBERO. requirements.txt'in tamamını DEĞİL, eval için gerekenleri kuruyoruz
#    (torch 2.4 / flash-attn / deepspeed / dm-reverb yok). mujoco 3.8.1 VLANeXt'in kendi pinidir; Linux'ta robosuite 1.4.1 ile çalışır.
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "clone ok"
%cd /content/VLANeXt
!mkdir -p third_party && git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git third_party/LIBERO
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" bddl easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds 2>&1 | tail -2
!pip install -q ./third_party/LIBERO 2>&1 | tail -1
# LIBERO ilk import'ta "custom path? (Y/N)" sorar; N cevabını borudan veriyoruz
!printf 'N\n' | python -c "import libero.libero; print('LIBERO ok')"

In [ ]:
# 4) T4 yamaları: flash_attention_2 → sdpa, torch.load mmap=True, VLANEXT_TASK_IDS görev filtresi
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py
!curl -s -o config/t4_bench_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/t4_bench_config.yaml
!python vlanext_t4_patch.py /content/VLANeXt

In [ ]:
# 5) Checkpoint (16.4 GB; Spatial suite için eğitilmiş). Qwen3-VL-2B omurgası ilk model yüklemesinde otomatik iner (~4.5 GB).
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints
!ls -lh checkpoints

In [ ]:
# 6) Eval: yalnızca görev 0, 2 ve 8 (between / table center / next to the plate), görev başına 1 episode.
#    İlk model yüklemesi birkaç dakika sürer (Qwen indirme + 16.4 GB state_dict kopyalama).
import os
os.environ["VLANEXT_TASK_IDS"] = "0,2,8"
os.environ["PYTHONPATH"] = "/content/VLANeXt/third_party/LIBERO"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ.pop("LIBERO_CONFIG_PATH", None)     # COMMON_ISSUES #6
!CUDA_VISIBLE_DEVICES=0 python -m scripts.libero_bench_eval --config config/t4_bench_config.yaml 2>&1 | grep -v Warning | tee eval_stdout.txt

In [ ]:
# 7) Sonuç: log'u tabloya çevir, runs.csv satırı üret, videoları listele
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/parse_eval_log.py
import glob
logs = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*/eval.log"))
print(logs)
!python parse_eval_log.py {logs[-1]} --run-id vlanext_spatial_t4_01 --csv results/runs.csv
!ls checkpoints/libero_spatial_checkpoint_*/*.mp4

In [ ]:
# 8) Bir rollout videosunu notebook içinde izle (başarısız olanı seç: nerede takılıyor?)
from IPython.display import Video
import glob
vids = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*/*.mp4"))
Video(vids[0], embed=True, width=384)

## Sonra
- `results/runs.csv` ve `eval.log`'u indir (Dosyalar paneli) → repoda `projects/` altında bir deney klasörüne koy.
- Ders 13'te bu kodun içini okuyacağız: komut nerede giriyor (`spec.task_description`), action nasıl decode ediliyor (`_denormalize_action`), success nerede hesaplanıyor (`env.step` → `done`).
- Türkçe deney için değiştirilecek tek satır `build_episode_specs` içindeki `task_description = task.language` — ama onu Ders 18'de yapacağız.